In [1]:
# Thiết lập chung
ROOT_DATA = "/kaggle/input/datasets/muhammadzahraan/3d-mri-scans-for-alzheimer-disease"
DATA_PROCESSED = "/kaggle/input/datasets/hngphm007/mri-2d-dataset"
OUTPUT = "/kaggle/working"

# "axial", "coronal", "sagittal", "All"
VIEWS_TO_USE = "coronal"
VIEW_CONFIG = {
    "axial": {"slices": 16, "start_frac": 0.15, "end_frac": 0.85, "target_size": (224, 224)},
    "coronal": {"slices": 32, "start_frac": 0.15, "end_frac": 0.85, "target_size": (224, 224)}, 
    "sagittal": {"slices": 32, "start_frac": 0.15, "end_frac": 0.85, "target_size": (224, 224)}  
}


NUM_EPOCHS = 60
EARLY_STOPPING = 20
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
IMAGE_SIZE = 224
NUM_FOLDS = 5
ACC_RATIO = 0.65
TRAINABLE_LAYERS = ["denseblock4", "norm5", "classifier"]

In [2]:
!rm -rf /kaggle/working/*

In [3]:
# pip install torch torchvision torchaudio scikit-learn matplotlib pillow opencv-python tqdm SimpleITK tensorboard imageio

In [4]:
# =========================
# Standard Library
# =========================
import os
import shutil
import zipfile
import pickle
import imageio
from argparse import ArgumentParser
from collections import Counter

# =========================
# Third-party: Numeric / Plot
# =========================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# =========================
# Image / Medical Imaging
# =========================
import cv2
import SimpleITK as sitk
from PIL import Image

# =========================
# PyTorch Core
# =========================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter

# =========================
# Torchvision
# =========================
from torchvision.datasets import CIFAR10
from torchvision.models import (
    resnet50,
    ResNet50_Weights,
    efficientnet_b0,
    EfficientNet_B0_Weights,
    convnext_tiny,           
    ConvNeXt_Tiny_Weights,
    densenet121, 
    DenseNet121_Weights
)

from torchvision.transforms import (
    Compose,
    Resize,
    ToTensor,
    Normalize,
    ColorJitter,
    RandomAffine
)

# =========================
# sklearn
# =========================
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix
)

# =========================
# Progress bar
# =========================
from tqdm.autonotebook import tqdm

2026-03-05 04:03:48.939031: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772683429.122459      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772683429.174435      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772683429.626632      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772683429.626671      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772683429.626673      55 computation_placer.cc:177] computation placer alr

# Processed_data.py

In [5]:
# # Xóa folder có sẵn
# ad_path = "/kaggle/working/AD"
# cn_path = "/kaggle/working/CN"
# zipp = "/kaggle/working/MRI_2D_dataset.zip"

# if os.path.exists(ad_path):
#     shutil.rmtree(ad_path)

# if os.path.exists(cn_path):
#     shutil.rmtree(cn_path)


# if os.path.exists(zipp):
#     os.remove(zipp)
    
# print("Deleted AD and CN and zip folders.")

In [6]:
# root_data = ROOT_DATA
# output_root = "/kaggle/working"

# AD_output = os.path.join(output_root, "AD")
# CN_output = os.path.join(output_root, "CN")

# os.makedirs(AD_output, exist_ok=True)
# os.makedirs(CN_output, exist_ok=True)

# AD_list = []
# CN_list = []

# seen_AD = set()
# seen_CN = set()


# for root, dirs, files in os.walk(root_data):
#     for file in files:
#         if file.endswith((".nii", ".nii.gz")):

#             full_path = os.path.join(root, file)
#             parts = root.split(os.sep)

#             subject_id = None
#             for p in parts:
#                 if "_S_" in p:
#                     subject_id = p
#                     break

#             if subject_id is None:
#                 continue

#             if "abb" in parts:
#                 if subject_id not in seen_AD:
#                     AD_list.append(full_path)
#                     seen_AD.add(subject_id)

#             elif "bbc" in parts or "ecc" in parts:
#                 if subject_id not in seen_CN:
#                     CN_list.append(full_path)
#                     seen_CN.add(subject_id)


# def resample_isotropic(img, spacing=(1.0, 1.0, 1.0)):

#     original_spacing = img.GetSpacing()
#     original_size = img.GetSize()

#     new_size = [
#         int(round(original_size[i] * original_spacing[i] / spacing[i]))
#         for i in range(3)
#     ]

#     resampled = sitk.Resample(
#         img,
#         new_size,
#         sitk.Transform(),
#         sitk.sitkLinear,
#         img.GetOrigin(),
#         spacing,
#         img.GetDirection(),
#         0,
#         img.GetPixelID()
#     )

#     return resampled



# def normalize_to_uint8(img):
#     img = img.astype(np.float32)
#     img = (img - img.min()) / (img.max() - img.min() + 1e-8)
#     img = (img * 255).astype(np.uint8)
#     return img


# def skull_stripping(img, view=None):

#     # Tạo mask (vùng khác nền)
#     mask = img > 0

#     coords = np.argwhere(mask)

#     # Nếu slice toàn nền đen
#     if coords.size == 0:
#         return img

#     rmin, cmin = coords.min(axis=0)
#     rmax, cmax = coords.max(axis=0)

#     if view == "axial":
#         w = cmax - cmin + 1
#         rmin = rmin + int(0.15 * w)

#     elif view == "coronal":
#         w = cmax - cmin + 1
#         rmax = rmax - int(0.45 * w)

#     elif view == "sagittal":
#         h = rmax - rmin + 1
#         w = cmax - cmin + 1
#         rmax = rmax - int(0.33 * h)
#         cmin = cmin + int(0.18 * w)

#     result = np.zeros_like(img)

#     # Copy vùng giữ lại
#     result[rmin:rmax+1, cmin:cmax+1] = img[rmin:rmax+1, cmin:cmax+1]

#     return result

    
# def process_list(file_list, class_output_dir):

#     for path in file_list:

#         parts = path.split(os.sep)
#         subject_id = None
#         for p in parts:
#             if "_S_" in p:
#                 subject_id = p
#                 break

#         if subject_id is None:
#             continue

#         subject_folder = os.path.join(class_output_dir, subject_id)
#         os.makedirs(subject_folder, exist_ok=True)

#         img = sitk.ReadImage(path)
#         img = resample_isotropic(img)

#         volume = sitk.GetArrayFromImage(img)
#         volume = np.transpose(volume, (2, 1, 0))  # (X,Y,Z)

#         X, Y, Z = volume.shape

#         # Axial (trục X) 
#         axial_indices = np.linspace(
#             VIEW_CONFIG["axial"]["start_frac"] * X,
#             VIEW_CONFIG["axial"]["end_frac"] * X,
#             VIEW_CONFIG["axial"]["slices"]
#         ).astype(int)

#         # Coronal (trục Y)
#         coronal_indices = np.linspace(
#             VIEW_CONFIG["coronal"]["start_frac"] * Y,
#             VIEW_CONFIG["coronal"]["end_frac"] * Y,
#             VIEW_CONFIG["coronal"]["slices"]
#         ).astype(int)

#         # Sagittal (trục Z)
#         sagittal_indices = np.linspace(
#             VIEW_CONFIG["sagittal"]["start_frac"] * Z,
#             VIEW_CONFIG["sagittal"]["end_frac"] * Z,
#             VIEW_CONFIG["sagittal"]["slices"]
#         ).astype(int)

#         # AXIAL
#         for i, idx in enumerate(axial_indices, 1):
#             slice_img = volume[idx, :, :]
#             slice_img = normalize_to_uint8(slice_img)
#             slice_img = skull_stripping(slice_img, "axial") 
#             Image.fromarray(slice_img).save(
#                 os.path.join(subject_folder, f"{subject_id}_axial_{i}.png")
#             )

#         # CORONAL
#         for i, idx in enumerate(coronal_indices, 1):
#             slice_img = volume[:, idx, :]
#             slice_img = normalize_to_uint8(slice_img)
#             slice_img = skull_stripping(slice_img, "coronal") 
#             Image.fromarray(slice_img).save(
#                 os.path.join(subject_folder, f"{subject_id}_coronal_{i}.png")
#             )

#         # SAGITTAL
#         for i, idx in enumerate(sagittal_indices, 1):
#             slice_img = volume[:, :, idx]
#             slice_img = normalize_to_uint8(slice_img)
#             slice_img = skull_stripping(slice_img, "sagittal") 
#             Image.fromarray(slice_img).save(
#                 os.path.join(subject_folder, f"{subject_id}_sagittal_{i}.png")
#             )


# process_list(AD_list, AD_output)
# process_list(CN_list, CN_output)



# zip_path = "/kaggle/working/MRI_2D_dataset.zip"

# with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
#     for root, dirs, files in os.walk(output_root):
#         for file in files:
#             if file.endswith(".png"):
#                 full_path = os.path.join(root, file)
#                 z.write(full_path, os.path.relpath(full_path, output_root))

# print("DONE.")
# print("Zip file location:", zip_path)
# # /kaggle/working/MRI_2D_dataset.zip

# Dataset.py

In [7]:
class MRIDataset(Dataset):
    def __init__(self, root="/kaggle/working", transform=None, view=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.id_patient = []
        self.categories = ["CN", "AD"]
        self.patients = {}

        for label, category in enumerate(self.categories):
            class_dir = os.path.join(root, category)

            for subject_id in os.listdir(class_dir):
                subject_path = os.path.join(class_dir, subject_id)

                patient_name = os.path.basename(subject_id)
                self.patients[patient_name] = label

                for file in os.listdir(subject_path):
                    if file.lower().endswith(".png"):

                        path = os.path.join(subject_path, file)

                        if view == "axial":
                            if "axial" in file.lower():
                                self.image_paths.append(path)
                                self.labels.append(label)
                                self.id_patient.append(patient_name)

                        elif view == "coronal":
                            if "coronal" in file.lower():
                                self.image_paths.append(path)
                                self.labels.append(label)
                                self.id_patient.append(patient_name)

                        elif view == "sagittal":
                            if "sagittal" in file.lower():
                                self.image_paths.append(path)
                                self.labels.append(label)
                                self.id_patient.append(patient_name)

                        elif view == "All":
                            self.image_paths.append(path)
                            self.labels.append(label)
                            self.id_patient.append(patient_name)

                        else:
                            raise ValueError("view must be: axial, coronal, sagittal or All")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        image_path = self.image_paths[index]
        label = self.labels[index]
        patient_id = self.id_patient[index]
    
        image = Image.open(image_path).convert("RGB")
    
        if self.transform:
            image = self.transform(image)
    
        return image, label, patient_id

# Model.py

In [8]:
class MyEfficentNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        self.backbone.classifier= nn.Identity()  # Remove the original fully connected layer
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5, inplace=True),
            nn.Linear(1280, num_classes),
        )
    
    def forward(self, x):
        return self.classifier(self.backbone(x))

In [9]:
class MyResNet50(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.backbone = resnet50(weights=ResNet50_Weights.DEFAULT)
        self.backbone.fc = nn.Identity()  # bỏ fc gốc

        self.classifier = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)  # ra (batch, 2048)
        x = self.classifier(x)
        return x

In [10]:
class MyConvNeXt(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.backbone = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)
        
        # bỏ classifier gốc
        self.backbone.classifier = nn.Identity()

        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Linear(768, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)   # (B, 768, 1, 1)
        x = torch.flatten(x, 1)  
        x = self.classifier(x)
        return x

In [11]:
class MyDenseNet121(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()

        if pretrained:
            weights = DenseNet121_Weights.IMAGENET1K_V1
        else:
            weights = None

        self.backbone = densenet121(weights=weights)

        # Lấy số feature cuối (1024)
        in_features = self.backbone.classifier.in_features

        # Bỏ classifier gốc
        self.backbone.classifier = nn.Identity()

        # Head mới
        self.classifier = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.backbone(x)   # shape: (B, 1024)
        x = self.classifier(x)
        return x

In [12]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * focal_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

In [13]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer

        self.activations = None
        self.gradients = None

        # đăng ký hook
        self.fwd_handle = target_layer.register_forward_hook(self._save_activation)
        self.bwd_handle = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def generate(self, logits, class_idx, input_size):
        self.model.zero_grad()

        one_hot = torch.zeros_like(logits)
        one_hot[:, class_idx] = 1
        logits.backward(one_hot)

        weights = torch.mean(self.gradients, dim=(2, 3), keepdim=True)
        cam = torch.sum(weights * self.activations, dim=1)
        cam = torch.relu(cam).squeeze()
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        cam = cam.detach().cpu().numpy()

        H, W = input_size
        cam = cv2.resize(cam, (W, H))
        return cam

# Train.py

In [14]:
def get_args():
    parser = ArgumentParser(description="CNN training")
    parser.add_argument("--seed", type=int, default=7)
    parser.add_argument("--root", "-r", type=str, default=DATA_PROCESSED, help="Root of the dataset")
    parser.add_argument("--folds", "-f", type=int, default=NUM_FOLDS, help="Number of folds")
    parser.add_argument("--epochs", "-e", type=int, default=NUM_EPOCHS, help="Number of epochs")
    parser.add_argument("--weight_decay", type=float, default=WEIGHT_DECAY)
    parser.add_argument("--early_stopping", "-s", type=int, default=EARLY_STOPPING)
    parser.add_argument("--batch_size", "-b", type=int, default=BATCH_SIZE, help="Batch size")
    parser.add_argument("--learning_rate", type=float, default=LEARNING_RATE)
    parser.add_argument("--image_size", "-i", type=int, default=IMAGE_SIZE, help="Image size")
    parser.add_argument("--logging", "-l", type=str, default=f"{OUTPUT}/tensorboard")
    parser.add_argument("--trained_models", "-t", type=str, default=f"{OUTPUT}/trained_models")
    parser.add_argument("--checkpoint", "-c", type=str, default=None)
    args = parser.parse_args(args=[])
    return args

In [15]:
def visualize_cam(patient_id, model, dataset, device, save_dir):

    model.eval()

    target_layer = model.backbone.features[-1]
    camger = GradCAM(model, target_layer)

    # bucket theo view
    frames_dict = {}

    patient_indices = [
        i for i, pid in enumerate(dataset.id_patient)
        if pid == patient_id
    ]

    for idx in patient_indices:

        # Lấy thông tin slice
        img_path = dataset.image_paths[idx]
        filename = os.path.basename(img_path)
        
        # Xác định view từ tên file (axial, coronal, sagittal)
        if 'axial' in filename.lower():
            view_name = 'axial'
        elif 'coronal' in filename.lower():
            view_name = 'coronal'
        elif 'sagittal' in filename.lower():
            view_name = 'sagittal'
        else:
            view_name = 'unknown'
            
        img, label, _ = dataset[idx]
        x = img.unsqueeze(0).to(device)

        logits = model(x)
        prob = torch.softmax(logits, dim=1)[0,1].item()

        cam = camger.generate(logits, class_idx=1, input_size=(x.shape[2], x.shape[3]))

        # unnormalize
        mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
        std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
        img_vis = img * std + mean
        img_vis = img_vis.permute(1,2,0).cpu().numpy()

        heatmap = cv2.applyColorMap(np.uint8(cam*255), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)/255.0

        overlay = 0.6*img_vis + 0.4*heatmap
        overlay = np.clip(overlay,0,1)

        frame = (overlay*255).astype(np.uint8)

        text = f"{view_name.upper()} | Score: {prob:.4f}"
        cv2.putText(
            frame,
            text,
            (20,30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255,255,255),
            2,
            cv2.LINE_AA
        )

        # tạo bucket nếu chưa tồn tại
        if view_name not in frames_dict:
            frames_dict[view_name] = []

        frames_dict[view_name].append(frame)

    # save mỗi view thành 1 GIF
    for view_name, frames in frames_dict.items():
        save_path = f"{save_dir}/{patient_id}_{view_name}.gif"
        imageio.mimsave(save_path, frames, fps=3)

In [16]:
def plot_confusion_matrix(writer, cm, class_names, epoch):

    cm_raw = cm.copy()

    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)
    cm_norm = np.round(cm_norm, 2)

    plt.style.use("seaborn-v0_8-white")
    figure = plt.figure(figsize=(8, 8), dpi=200)

    ax = sns.heatmap(
        cm_norm,
        annot=False,
        cmap="Blues",
        cbar=True,
        square=True,
        linewidths=1,
        linecolor='gray'
    )

    ax.set_xticklabels(class_names, rotation=45, fontsize=12)
    ax.set_yticklabels(class_names, rotation=0, fontsize=12)

    ax.set_ylabel("True Label", fontsize=14)
    ax.set_xlabel("Predicted Label", fontsize=14)
    ax.set_title("Confusion Matrix (Normalized)", fontsize=16, pad=20)

    threshold = cm_norm.max() / 2.0

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            text_color = "white" if cm_norm[i, j] > threshold else "black"
            ax.text(
                j + 0.5,
                i + 0.5,
                f"{cm_raw[i, j]}\n({cm_norm[i, j]:.2f})",
                ha="center",
                va="center",
                color=text_color,
                fontsize=12,
                fontweight="bold"
            )

    plt.tight_layout()

    writer.add_figure("Confusion_Matrix", figure, epoch)
    plt.show()
    plt.close(figure)

In [17]:
def plot_metrics(
    writer,
    train_losses,
    val_acc,
    val_auc,
    val_labels,
    val_probs,
    class_names,
    epoch,
    tag="",
    save_dir=None
):
    """
    Vẽ 5 metric trong 1 figure:

    Row 1:
        - Train Loss
        - Val Accuracy
        - Val AUC

    Row 2:
        - Confusion Matrix
        - ROC Curve
        - (empty)
    """

    epochs = list(range(1, len(train_losses) + 1))

    fig, axes = plt.subplots(2, 3, figsize=(18, 10), dpi=200)

    # =======================
    # ROW 1
    # =======================

    # Train Loss
    axes[0, 0].plot(epochs, train_losses, marker='o', linewidth=2)
    axes[0, 0].set_title(f"{tag} - Train Loss")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].grid(True)

    # Val Accuracy
    axes[0, 1].plot(epochs, val_acc, marker='s', linewidth=2)
    axes[0, 1].set_title(f"{tag} - Val Accuracy")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("Accuracy")
    axes[0, 1].set_ylim(0, 1)
    axes[0, 1].grid(True)

    # Val AUC
    axes[0, 2].plot(epochs, val_auc, marker='^', linewidth=2)
    axes[0, 2].set_title(f"{tag} - Val AUC")
    axes[0, 2].set_xlabel("Epoch")
    axes[0, 2].set_ylabel("AUC")
    axes[0, 2].set_ylim(0, 1)
    axes[0, 2].grid(True)

    # =======================
    # ROW 2
    # =======================

    # Confusion Matrix (best threshold = 0.5)
    preds_bin = [1 if p >= 0.5 else 0 for p in val_probs]
    cm = confusion_matrix(val_labels, preds_bin)

    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    cm_norm = np.nan_to_num(cm_norm)
    cm_norm = np.round(cm_norm, 2)

    sns.heatmap(
        cm_norm,
        annot=False,
        cmap="Blues",
        cbar=True,
        square=True,
        linewidths=1,
        linecolor='gray',
        ax=axes[1, 0]
    )

    axes[1, 0].set_title(f"{tag} - Confusion Matrix")
    axes[1, 0].set_xlabel("Predicted")
    axes[1, 0].set_ylabel("True")
    axes[1, 0].set_xticklabels(class_names, rotation=45)
    axes[1, 0].set_yticklabels(class_names, rotation=0)

    threshold = cm_norm.max() / 2.0

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = "white" if cm_norm[i, j] > threshold else "black"
            axes[1, 0].text(
                j + 0.5,
                i + 0.5,
                f"{cm[i,j]}\n({cm_norm[i,j]:.2f})",
                ha="center",
                va="center",
                color=color,
                fontsize=11,
                fontweight="bold"
            )

    # ROC Curve
    try:
        fpr, tpr, _ = roc_curve(val_labels, val_probs)
        auc_score = roc_auc_score(val_labels, val_probs)
        axes[1, 1].plot(fpr, tpr, linewidth=2, label=f"AUC = {auc_score:.4f}")
    except ValueError:
        axes[1, 1].text(0.5, 0.5, "AUC N/A", ha='center', va='center')

    axes[1, 1].plot([0, 1], [0, 1], linestyle="--")
    axes[1, 1].set_title(f"{tag} - ROC Curve")
    axes[1, 1].set_xlabel("FPR")
    axes[1, 1].set_ylabel("TPR")
    axes[1, 1].legend()
    axes[1, 1].grid(True)

    # Hide last subplot (2,3)
    axes[1, 2].axis("off")

    plt.tight_layout()

    if save_dir is not None:
        os.makedirs(save_dir, exist_ok=True)
        fname = os.path.join(save_dir, f"{tag}_all_metrics.png".replace(" ", "_"))
        fig.savefig(fname, dpi=200)

    writer.add_figure(f"{tag}/All_Metrics", fig, epoch)

    plt.show()
    plt.close(fig)

In [18]:
def collect_fold_cams(val_patients, model, full_dataset, device, save_dir):

    model.eval()

    target_layer = model.backbone.features[-1]
    camger = GradCAM(model, target_layer)

    views = ["axial", "coronal", "sagittal"] if VIEWS_TO_USE == "All" else [VIEWS_TO_USE]
    val_patients_set = set(val_patients)

    # Gom tất cả indices của val patients
    val_indices = [
        i for i, pid in enumerate(full_dataset.id_patient)
        if pid in val_patients_set
    ]

    # Batch forward để lấy scores (không cần grad)
    all_imgs = torch.stack([full_dataset[i][0] for i in val_indices]).to(device)

    chunk_size = 64
    all_probs = []
    with torch.no_grad():
        for i in range(0, len(all_imgs), chunk_size):
            chunk = all_imgs[i:i + chunk_size]
            logits = model(chunk)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().tolist()
            all_probs.extend(probs)

    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    for local_idx, global_idx in enumerate(val_indices):

        pid = full_dataset.id_patient[global_idx]
        image_path = full_dataset.image_paths[global_idx]

        view_name = next((v for v in views if v in image_path.lower()), None)
        if view_name is None:
            continue

        # Tạo folder riêng cho từng patient
        patient_dir = os.path.join(save_dir, pid)
        os.makedirs(patient_dir, exist_ok=True)

        x = all_imgs[local_idx].unsqueeze(0)

        logits = model(x)
        prob = all_probs[local_idx]

        cam = camger.generate(logits, class_idx=1, input_size=(x.shape[2], x.shape[3]))

        # Unnormalize ảnh gốc
        img = all_imgs[local_idx].cpu()
        img_vis = (img * std + mean).permute(1, 2, 0).numpy()

        heatmap = cv2.applyColorMap(np.uint8(cam * 255), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0

        overlay = np.clip(0.6 * img_vis + 0.4 * heatmap, 0, 1)
        frame = (overlay * 255).astype(np.uint8)

        text = f"{view_name.upper()} | Score: {prob:.4f}"

        h, w = frame.shape[:2]
        
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.5     # mặc định nhỏ gọn
        thickness = 1
        
        # đo kích thước text
        (text_w, text_h), _ = cv2.getTextSize(text, font, font_scale, thickness)
        
        # nếu text dài hơn ảnh -> co lại vừa khung
        if text_w > w - 20:
            font_scale *= (w - 20) / text_w
            (text_w, text_h), _ = cv2.getTextSize(text, font, font_scale, thickness)
        
        # vẽ text (luôn nằm trong ảnh)
        cv2.putText(
            frame,
            text,
            (10, 10 + text_h),
            font,
            font_scale,
            (255, 255, 255),
            thickness,
            cv2.LINE_AA
        )

        # Lấy tên file gốc làm tên file PNG heatmap
        slice_name = os.path.splitext(os.path.basename(image_path))[0]
        save_path = os.path.join(patient_dir, f"{slice_name}_cam.png")
        Image.fromarray(frame).save(save_path)

    print(f"[GradCAM] Saved heatmaps -> {save_dir}")

In [19]:
args = get_args()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = Compose([
    RandomAffine(degrees=(-5, 5), 
                         translate=(0.05, 0.05), 
                         scale=(0.85, 1.15), 
                         shear=5
    ),
    ColorJitter(brightness=(1.3, 1.3)),
    Resize((args.image_size, args.image_size)),
    ToTensor(),
    Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

full_dataset = MRIDataset(root=args.root, transform=transform, view=VIEWS_TO_USE)

indices = list(full_dataset.patients.keys())
labels = [full_dataset.patients[p] for p in indices]

indices = np.array(indices)
labels = np.array(labels)


train_val_patients, test_patients, train_val_labels, test_labels = train_test_split(
    indices,
    labels,
    test_size=0.1,
    stratify=labels,
    random_state=args.seed
)

# Convert patient -> image index
train_val_idx = [
    i for i, pid in enumerate(full_dataset.id_patient)
    if pid in train_val_patients
]

test_idx = [
    i for i, pid in enumerate(full_dataset.id_patient)
    if pid in test_patients
]


if os.path.isdir(args.logging):
    shutil.rmtree(args.logging)

if not os.path.isdir(args.trained_models):
    os.mkdir(args.trained_models)

writer = SummaryWriter(args.logging)

skf = StratifiedKFold(
    n_splits=args.folds,
    shuffle=True,
    random_state=args.seed
)


fold_accuracies = []
fold_precisions = []
fold_recalls = []
fold_f1s = []


for fold, (train_p_rel, val_p_rel) in enumerate(skf.split(train_val_patients, train_val_labels)):

    print(f"\n========== FOLD {fold+1}/{args.folds} ==========")

    train_patients = train_val_patients[train_p_rel]
    val_patients = train_val_patients[val_p_rel]

    train_patients = set(train_patients) # Tối ưu truy vấn
    val_patients   = set(val_patients)
    
    # Convert patient -> image index
    train_idx = [
        i for i, pid in enumerate(full_dataset.id_patient)
        if pid in train_patients
    ]

    val_idx = [
        i for i, pid in enumerate(full_dataset.id_patient)
        if pid in val_patients
    ]

    train_dataset = Subset(full_dataset, train_idx)
    val_dataset = Subset(full_dataset, val_idx)

    train_dataloader = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=4,
        drop_last=True
    )

    val_dataloader = DataLoader(
        val_dataset,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=4,
        drop_last=False
    )

    # Lấy label của train samples
    train_labels_fold = [
        full_dataset.labels[i]  # hoặc full_dataset.targets[i]
        for i in train_idx
    ]
    
    counter = Counter(train_labels_fold)
    
    total = sum(counter.values())
    
    # Công thức chuẩn:
    # weight_c = total_samples / (num_classes * num_samples_class_c)
    num_classes = 2
    
    class_weights = []
    for c in range(num_classes):
        class_weights.append(total / (num_classes * counter[c]))
    
    class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)


    
    model = MyDenseNet121(num_classes=num_classes).to(device)

    trainable_layers = TRAINABLE_LAYERS

    for name, param in model.named_parameters():
        if any(layer in name for layer in trainable_layers):
            param.requires_grad = True
        else:
            param.requires_grad = False

    # criterion = nn.CrossEntropyLoss(weight=class_weights)

    alpha = class_weights  
    criterion = FocalLoss(alpha=alpha, gamma=2)

    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=args.learning_rate,
        weight_decay=args.weight_decay
    )


    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=args.epochs
    )

    best_acc = 0
    best_score = 0
    best_precision = 0
    best_recall = 0
    best_f1 = 0
    best_auc = 0
    best_cm = None
    best_epoch = 0
    best_val_labels = []
    best_val_probs = []

    es_counter = 0          # số epoch liên tiếp không cải thiện
    
    num_iters = len(train_dataloader)


    epoch_train_losses = []   # trung bình loss mỗi epoch
    epoch_val_acc = []        # accuracy mỗi epoch
    epoch_val_auc = []

    
    for epoch in range(args.epochs):

        running_loss = 0.0

        model.train()
        progress_bar = tqdm(train_dataloader, colour="green")

        for iter, (images, labels_batch, _) in enumerate(progress_bar):

            images = images.to(device)
            labels_batch = labels_batch.to(device)

            outputs = model(images)
            loss_value = criterion(outputs, labels_batch)

            running_loss += loss_value.item()

            progress_bar.set_description(
                f"Fold {fold+1} | "
                f"Epoch {epoch+1}/{args.epochs} | "
                f"Iter {iter+1}/{num_iters} | "
                f"Loss {loss_value.item():.3f}"
            )

            global_step = fold * args.epochs * num_iters + epoch * num_iters + iter
            writer.add_scalar("Train/Loss", loss_value.item(), global_step)

            optimizer.zero_grad()
            loss_value.backward()
            optimizer.step()

        epoch_loss = running_loss / float(num_iters) if num_iters > 0 else running_loss
        epoch_train_losses.append(epoch_loss)
        writer.add_scalar(f"Fold_{fold+1}/Train_Loss", epoch_loss, epoch)

        model.eval()

        # dict gom prob và label theo patient
        pat_probs = {}   # patient_id -> list of probs
        pat_labels = {}  # patient_id -> ground truth label
        
        with torch.no_grad():
            for images, labels_batch, patient_ids in val_dataloader:
                images = images.to(device)
        
                outputs = model(images)
                probs = torch.softmax(outputs, dim=1)[:, 1]
        
                for pid, prob, lbl in zip(patient_ids, probs.cpu().tolist(), labels_batch.tolist()):
                    if pid not in pat_probs:
                        pat_probs[pid] = []
                        pat_labels[pid] = lbl
                    pat_probs[pid].append(prob)
        
        # Aggregate: mean prob của tất cả slices -> quyết định nhãn patient
        all_probs       = [sum(pat_probs[pid]) / len(pat_probs[pid]) for pid in pat_probs]
        all_labels      = [pat_labels[pid] for pid in pat_probs]
        all_predictions = [1 if p >= 0.5 else 0 for p in all_probs]

        accuracy = accuracy_score(all_labels, all_predictions)
        precision = precision_score(all_labels, all_predictions, zero_division=0)
        recall = recall_score(all_labels, all_predictions, zero_division=0)
        f1 = f1_score(all_labels, all_predictions, zero_division=0)

        cm = confusion_matrix(all_labels, all_predictions)

        try:
            auc = roc_auc_score(all_labels, all_probs)
        except ValueError:
            auc = float('nan')

        score = ACC_RATIO * accuracy + (1 - ACC_RATIO) * recall

        print(
            f"Fold {fold+1} | Epoch {epoch+1} | "
            f"Accuracy: {accuracy:.4f} | "
            f"Precision: {precision:.4f} | "
            f"Recall: {recall:.4f} | "
            f"F1: {f1:.4f} | AUC: {auc if not np.isnan(auc) else 'nan'}"
        )

        writer.add_scalar(f"Fold_{fold+1}/Val_Accuracy", accuracy, epoch)
        writer.add_scalar(f"Fold_{fold+1}/Val_Precision", precision, epoch)
        writer.add_scalar(f"Fold_{fold+1}/Val_Recall", recall, epoch)
        writer.add_scalar(f"Fold_{fold+1}/Val_F1", f1, epoch)
        writer.add_scalar(f"Fold_{fold+1}/Val_AUC", auc if not np.isnan(auc) else -1.0, epoch)

        epoch_val_acc.append(accuracy)
        epoch_val_auc.append(auc if not np.isnan(auc) else 0.0)


        scheduler.step()

        # SAVE LAST
        checkpoint = {
            "epoch": epoch + 1,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict()
        }
        torch.save(checkpoint, f"{args.trained_models}/fold_{fold+1}_last.pt")

        # SAVE BEST + Early Stopping
        if score > best_score:
            best_acc = accuracy
            best_precision = precision
            best_recall = recall
            best_f1 = f1
            best_auc = auc if not np.isnan(auc) else 0.0
            best_cm = cm.copy()
            best_epoch = epoch + 1
            best_val_labels = all_labels.copy()
            best_val_probs = all_probs.copy()
            best_score = score

            checkpoint = {
                "epoch": epoch + 1,
                "best_acc": best_acc,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict()
            }
            torch.save(checkpoint, f"{args.trained_models}/fold_{fold+1}_best.pt")

            es_counter = 0   # reset early stopping counter
        else:
            es_counter += 1
            if es_counter >= args.early_stopping:
                print(f"  [Early Stopping] No improvement for {args.early_stopping} epochs. Stop fold {fold+1}.")
                break

    fold_accuracies.append(best_acc)
    fold_precisions.append(best_precision)
    fold_recalls.append(best_recall)
    fold_f1s.append(best_f1)
    
    print(f"\n===== BEST RESULT FOLD {fold+1} =====")
    print(f"Best Epoch : {best_epoch}")
    print(f"Accuracy   : {best_acc:.4f}")
    print(f"Precision  : {best_precision:.4f}")
    print(f"Recall     : {best_recall:.4f}")
    print(f"F1         : {best_f1:.4f}")
    print(f"AUC        : {best_auc:.4f}")
    print("Best Confusion Matrix:\n", best_cm)

    
    plot_metrics(
        writer=writer,
        train_losses=epoch_train_losses,
        val_acc=epoch_val_acc,
        val_auc=epoch_val_auc,
        val_labels=best_val_labels,
        val_probs=best_val_probs,
        class_names=full_dataset.categories,
        epoch=fold,
        tag=f"Fold_{fold+1}",
        save_dir=OUTPUT
    )


    # # Load best model weights trước
    # ckpt = torch.load(f"{args.trained_models}/fold_{fold+1}_best.pt")
    # model.load_state_dict(ckpt["model"])
    # model.to(device)

    # cam_dir = os.path.join(OUTPUT, f"fold_{fold+1}_cams")
    # collect_fold_cams(val_patients, model, full_dataset, device, save_dir=cam_dir)


# ── Sau khi train tất cả folds: best overall ────────────────
best_fold_idx = int(np.argmax(fold_accuracies))

print(f"\n========== BEST MODEL (Fold {best_fold_idx+1}) ==========")
print(f"Accuracy   : {fold_accuracies[best_fold_idx]:.4f}")
print(f"Precision  : {fold_precisions[best_fold_idx]:.4f}")
print(f"Recall     : {fold_recalls[best_fold_idx]:.4f}")
print(f"F1         : {fold_f1s[best_fold_idx]:.4f}")


========== FOLD 1/5 ==========
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 207MB/s]


  0%|          | 0/516 [00:00<?, ?it/s]

Fold 1 | Epoch 1 | Accuracy: 0.6667 | Precision: 0.3725 | Recall: 0.6333 | F1: 0.4691 | AUC: 0.7097643097643097


  0%|          | 0/516 [00:00<?, ?it/s]

...

Fold 5 | Epoch 27 | Accuracy: 0.7364 | Precision: 0.4074 | Recall: 0.3793 | F1: 0.3929 | AUC: 0.6686206896551724


  0%|          | 0/516 [00:00<?, ?it/s]

Fold 5 | Epoch 28 | Accuracy: 0.7519 | Precision: 0.4483 | Recall: 0.4483 | F1: 0.4483 | AUC: 0.6403448275862069


  0%|          | 0/516 [00:00<?, ?it/s]

Fold 5 | Epoch 29 | Accuracy: 0.7287 | Precision: 0.3750 | Recall: 0.3103 | F1: 0.3396 | AUC: 0.6527586206896552
  [Early Stopping] No improvement for 20 epochs. Stop fold 5.

===== BEST RESULT FOLD 5 =====
Best Epoch : 9
Accuracy   : 0.7829
Precision  : 0.5152
Recall     : 0.5862
F1         : 0.5484
AUC        : 0.7552
Best Confusion Matrix:
 [[84 16]
 [12 17]]

========== BEST MODEL (Fold 5) ==========
Accuracy   : 0.7829
Precision  : 0.5152
Recall     : 0.5862
F1         : 0.5484


In [20]:
ckpt = torch.load(f"{args.trained_models}/fold_1_best.pt")
model.load_state_dict(ckpt["model"])/kaggle/working/Fold_1_all_metrics.png
model.to(device)

visualize_cam(
    patient_id="123_S_4567",
    model=model,
    dataset=full_dataset,
    device=device,
    save_dir=f"{OUTPUT}/heatmap"
)

NameError: name 'kaggle' is not defined

# Test.py

In [ ]:
print("=== Test patients summary ===")

print("Total test patients:", len(test_patients))
print("Patient IDs:", list(test_patients))

print("\nPatient -> Class:")
for p in test_patients:
    label = full_dataset.patients[p]
    class_name = full_dataset.categories[label]
    print(p, "->", class_name)

print("=== End test patients summary ===")

In [ ]:
# categories = ["CN", "AD"]

# test_dataset = Subset(full_dataset, test_idx)

# test_dataloader = DataLoader(
#     test_dataset,
#     batch_size=args.batch_size,
#     shuffle=False,
#     num_workers=4,
#     drop_last=False
# )

# # Load model
# model = MyDenseNet121(num_classes=2).to(device)

# if args.checkpoint:
#     checkpoint = torch.load(args.checkpoint)
#     model.load_state_dict(checkpoint["model"])
# else:
#     print("No checkpoint found!")
#     exit(0)

# model.eval()


# pat_probs  = {}
# pat_labels = {}

# with torch.no_grad():
#     for images, labels, patient_ids in test_dataloader:
#         images = images.to(device)

#         outputs = model(images)
#         probs = torch.softmax(outputs, dim=1)[:, 1]

#         for pid, prob, lbl in zip(patient_ids, probs.cpu().tolist(), labels.tolist()):
#             if pid not in pat_probs:
#                 pat_probs[pid] = []
#                 pat_labels[pid] = lbl
#             pat_probs[pid].append(prob)

# all_probs       = [sum(pat_probs[pid]) / len(pat_probs[pid]) for pid in pat_probs]
# all_labels      = [pat_labels[pid] for pid in pat_probs]
# all_predictions = [1 if p >= 0.5 else 0 for p in all_probs]



# accuracy = accuracy_score(all_labels, all_predictions)
# precision = precision_score(all_labels, all_predictions, zero_division=0)
# recall = recall_score(all_labels, all_predictions, zero_division=0)
# f1 = f1_score(all_labels, all_predictions, zero_division=0)

# try:
#     auc = roc_auc_score(all_labels, all_probs)
# except ValueError:
#     auc = float("nan")

# print(
#     f"\n========== TEST RESULT ==========\n"
#     f"Accuracy : {accuracy:.4f}\n"
#     f"Precision: {precision:.4f}\n"
#     f"Recall   : {recall:.4f}\n"
#     f"F1       : {f1:.4f}\n"
#     f"AUC      : {auc if not np.isnan(auc) else 'nan'}"
# )

# # Log scalar
# writer.add_scalar("Test/Accuracy", accuracy)
# writer.add_scalar("Test/Precision", precision)
# writer.add_scalar("Test/Recall", recall)
# writer.add_scalar("Test/F1", f1)
# writer.add_scalar("Test/AUC", auc if not np.isnan(auc) else -1.0)


# plot_metrics(
#     writer=writer,
#     train_losses=[0.0],             
#     val_acc=[accuracy],
#     val_auc=[auc if not np.isnan(auc) else 0.0],
#     val_labels=all_labels,
#     val_probs=all_probs,
#     class_names=categories,
#     epoch=0,
#     tag="Test",
#     save_dir=OUTPUT
# )

# writer.close()